In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from keras.src.ops import dtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [4]:
df = pd.read_csv('Customer_Churn_Data_V3.csv')

In [5]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Contract_Month-to-month,Contract_One year,Contract_Two year,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [6]:
features = df.drop(columns = ['Churn'])
target = df['Churn']

In [7]:
features_train,features_test,target_train,target_test = train_test_split(features,target,test_size=0.2,random_state=42)

In [8]:
smote = SMOTE(random_state=42)
features_train,target_train = smote.fit_resample(features_train,target_train)

In [9]:
scaler = StandardScaler()
features_train = scaler.fit_transform(features_train)
features_test = scaler.transform(features_test)

In [10]:
features_train_tensor = torch.tensor(features_train, dtype=torch.float32)
features_test_tensor = torch.tensor(features_test, dtype=torch.float32)
target_train_tensor = torch.tensor(target_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
target_test_tensor = torch.tensor(target_test.to_numpy(), dtype=torch.float32).reshape(-1, 1)

In [11]:
train_dataset = TensorDataset(features_train_tensor, target_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(features_test_tensor, target_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32)

In [12]:
class ChurnANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(40, 11)
        self.fc2 = nn.Linear(11, 6)
        self.fc3 = nn.Linear(6, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

In [13]:
model = ChurnANN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
epochs = 50

for epoch in range(epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

Epoch 10/50 | Loss: 0.5333
Epoch 20/50 | Loss: 0.3509
Epoch 30/50 | Loss: 0.1995
Epoch 40/50 | Loss: 0.7797
Epoch 50/50 | Loss: 0.6540


In [15]:
model.eval()
with torch.no_grad():
    y_pred = model(features_test_tensor)
    y_pred_labels = (y_pred > 0.5).float()

from sklearn.metrics import confusion_matrix, accuracy_score
y_pred_np = y_pred_labels.numpy()
y_test_np = target_test_tensor.numpy()

print(accuracy_score(y_test_np, y_pred_np))
print(confusion_matrix(y_test_np, y_pred_np))

0.7420042643923241
[[792 241]
 [122 252]]


In [16]:
print("/n")

/n
